# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Outliers

Detect and handle outliers in your data using visual and statistical methods.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Create a dataset with outliers

In [ ]:
np.random.seed(42)
n = 300

df = pd.DataFrame({
    "price": np.concatenate([
        np.random.normal(250000, 40000, n - 5),
        [1_500_000, 1_200_000, -5000, 0, 999_999_999]
    ]),
    "sqft": np.concatenate([
        np.random.normal(1600, 350, n - 3),
        [8500, 9200, -100]
    ]),
    "age_years": np.concatenate([
        np.random.normal(18, 8, n - 2),
        [150, -5]
    ])
})

print("Dataset shape:", df.shape)

## Visual method: Box plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, col in zip(axes, ["price", "sqft", "age_years"]):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor="steelblue", alpha=0.6),
               medianprops=dict(color="red", linewidth=2),
               flierprops=dict(marker="o", color="red", alpha=0.5))
    ax.set_title(col)
    ax.set_ylabel("Value")

plt.suptitle("Box Plots — Visual Outlier Detection")
plt.tight_layout()
plt.show()

## Statistical method: IQR method

In [ ]:
def iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (series < lower) | (series > upper)
    print(f"\n{series.name}:")
    print(f"  Bounds: [{lower:,.1f}, {upper:,.1f}]")
    print(f"  Outliers: {mask.sum()} ({mask.mean()*100:.1f}%)")
    return mask

for col in ["price", "sqft", "age_years"]:
    iqr_outliers(df[col])

## Statistical method: Z-score method

In [ ]:
from scipy import stats

def z_score_outliers(series, threshold=3):
    z_scores = np.abs(stats.zscore(series.dropna()))
    mask = z_scores > threshold
    print(f"\n{series.name} (threshold={threshold}):")
    print(f"  Outliers: {mask.sum()} ({mask.mean()*100:.1f}%)")
    return mask

for col in ["price", "sqft", "age_years"]:
    z_score_outliers(df[col])

## Handling outliers: Winsorization (capping)

In [ ]:
def winsorize_column(series, lower_percentile=1, upper_percentile=99):
    lower_bound = series.quantile(lower_percentile / 100)
    upper_bound = series.quantile(upper_percentile / 100)
    winsorized = series.clip(lower_bound, upper_bound)
    return winsorized

df_winsorized = df.copy()
df_winsorized["price"] = winsorize_column(df["price"])

print("Original price range:", df["price"].min(), "to", df["price"].max())
print("Winsorized price range:", df_winsorized["price"].min(), "to", df_winsorized["price"].max())